# 🧠 Minería de Datos — Naive Bayes
## Notebook autónomo explicado para estudiantes

Este notebook introduce el modelo **Naive Bayes**, un clasificador probabilístico basado en el Teorema de Bayes.

Naive Bayes aparece dentro del bloque de clasificación del syllabus y permite entender un enfoque diferente a los modelos ya vistos:

- KNN clasifica por distancia.
- Árboles clasifican por reglas.
- Random Forest combina árboles.
- SVM busca una frontera de separación.
- Naive Bayes clasifica usando probabilidad.

---

## Objetivos

Al finalizar este notebook podrás:

1. Explicar qué es Naive Bayes.
2. Comprender el Teorema de Bayes.
3. Entender la idea de probabilidad condicional.
4. Comprender la suposición de independencia.
5. Entrenar un modelo Gaussian Naive Bayes.
6. Evaluar el modelo con métricas de clasificación.
7. Comparar Naive Bayes contra otros modelos.
8. Identificar ventajas y limitaciones.
9. Aplicar Naive Bayes a un dataset real.

# 1. Idea central de Naive Bayes

Naive Bayes es un modelo de clasificación probabilístico.

La pregunta que responde es:

> ¿Cuál es la clase más probable dado lo que observo en las variables?

Formalmente:

\[
P(Clase | Datos)
\]

Es decir:

> probabilidad de una clase dado un conjunto de características.

---

## Ejemplo intuitivo

Si una persona tiene ciertas características médicas, el modelo estima:

- probabilidad de clase A,
- probabilidad de clase B.

Luego asigna la clase con mayor probabilidad.

# 2. Teorema de Bayes

Naive Bayes se basa en el Teorema de Bayes:

\[
P(A|B)=\frac{P(B|A)P(A)}{P(B)}
\]

En clasificación:

\[
P(C|X)=\frac{P(X|C)P(C)}{P(X)}
\]

Donde:

- \(C\): clase.
- \(X\): variables observadas.
- \(P(C|X)\): probabilidad de la clase dado los datos.
- \(P(X|C)\): probabilidad de observar los datos si la clase es C.
- \(P(C)\): probabilidad previa de la clase.
- \(P(X)\): probabilidad de los datos.

---

## Idea clave

El modelo calcula qué clase es más probable.

# 3. ¿Por qué se llama 'Naive'?

La palabra `naive` significa ingenuo.

Se llama así porque el modelo asume que las variables son independientes entre sí dado el valor de la clase.

Esto rara vez se cumple perfectamente en la vida real.

Pero aun así, Naive Bayes puede funcionar sorprendentemente bien.

---

## Suposición principal

\[
P(X_1, X_2, ..., X_n | C) = P(X_1|C)P(X_2|C)...P(X_n|C)
\]

En palabras:

> el modelo trata cada variable como si aportara información independiente.

# 4. Tipos principales de Naive Bayes

| Variante | Uso común |
|---|---|
| GaussianNB | variables numéricas continuas |
| MultinomialNB | conteos, texto, frecuencias |
| BernoulliNB | variables binarias |

En este notebook usaremos:

# Gaussian Naive Bayes

porque trabajaremos con variables numéricas continuas.

# 5. Importación de librerías

Usaremos:

- `pandas` y `numpy` para datos.
- `matplotlib` para gráficos.
- `load_breast_cancer` como dataset real.
- `GaussianNB` para Naive Bayes.
- modelos adicionales para comparación.
- métricas de clasificación.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

from sklearn.naive_bayes import GaussianNB
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    ConfusionMatrixDisplay,
    classification_report
)

pd.set_option("display.max_columns", 80)
pd.set_option("display.width", 120)

# 6. Dataset real: Breast Cancer Wisconsin

Usaremos el mismo dataset médico que puede usarse en otros modelos de clasificación.

El objetivo es clasificar tumores en dos clases:

- malignant
- benign

Es un problema de clasificación binaria.

In [ ]:
data = load_breast_cancer()

X = pd.DataFrame(data.data, columns=data.feature_names)
y = pd.Series(data.target, name="target")

print("Dimensiones de X:", X.shape)
print("Dimensión de y:", y.shape)
print("Clases:", data.target_names)

X.head()

# 7. Exploración inicial

Antes de entrenar el modelo revisamos:

1. estructura,
2. valores faltantes,
3. distribución de clases.

In [ ]:
X.info()

In [ ]:
print("Total de valores faltantes:", X.isna().sum().sum())

In [ ]:
conteo = y.value_counts().sort_index()
porcentaje = y.value_counts(normalize=True).sort_index() * 100

resumen_clases = pd.DataFrame({
    "codigo": conteo.index,
    "clase": data.target_names,
    "conteo": conteo.values,
    "porcentaje": porcentaje.round(2).values
})

resumen_clases

In [ ]:
plt.figure(figsize=(6,4))
plt.bar(resumen_clases["clase"], resumen_clases["conteo"])
plt.title("Distribución de clases")
plt.xlabel("Clase")
plt.ylabel("Número de observaciones")
plt.show()

# 8. Separación train/test

Separamos los datos en:

- 70% entrenamiento.
- 30% prueba.

Usamos `stratify=y` para mantener proporciones similares de clases.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.30,
    random_state=42,
    stratify=y
)

print("Train:", X_train.shape)
print("Test:", X_test.shape)

print("\nDistribución train:")
print(y_train.value_counts(normalize=True).sort_index())

print("\nDistribución test:")
print(y_test.value_counts(normalize=True).sort_index())

# 9. Función de evaluación

Usaremos las mismas métricas de clasificación:

- accuracy,
- precision,
- recall,
- F1-score.

Esto permite comparar Naive Bayes contra otros modelos.

In [ ]:
def evaluar_clasificacion(nombre, y_real, y_pred):
    return {
        "modelo": nombre,
        "accuracy": accuracy_score(y_real, y_pred),
        "precision": precision_score(y_real, y_pred),
        "recall": recall_score(y_real, y_pred),
        "f1": f1_score(y_real, y_pred)
    }

# 10. Entrenamiento de Gaussian Naive Bayes

GaussianNB asume que las variables numéricas siguen una distribución normal dentro de cada clase.

No siempre se cumple exactamente, pero es una aproximación útil.

In [ ]:
modelo_nb = GaussianNB()

modelo_nb.fit(X_train, y_train)

pred_nb = modelo_nb.predict(X_test)

print(classification_report(y_test, pred_nb, target_names=data.target_names))

# 11. Matriz de confusión

La matriz de confusión permite ver:

- verdaderos positivos,
- verdaderos negativos,
- falsos positivos,
- falsos negativos.

En problemas médicos, los falsos negativos pueden ser especialmente delicados.

In [ ]:
cm = confusion_matrix(y_test, pred_nb)

disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=data.target_names
)

disp.plot()
plt.title("Matriz de confusión - Naive Bayes")
plt.show()

# 12. Probabilidades predichas

Naive Bayes puede entregar probabilidades para cada clase.

Esto permite analizar qué tan seguro está el modelo en sus predicciones.

In [ ]:
probabilidades = modelo_nb.predict_proba(X_test)

prob_df = pd.DataFrame(
    probabilidades,
    columns=[f"prob_{clase}" for clase in data.target_names]
)

prob_df.head()

# 13. Predicción de un caso individual

Tomemos un registro del conjunto de prueba y observemos:

- valores reales,
- probabilidades predichas,
- clase predicha.

In [ ]:
i = 0

caso = X_test.iloc[[i]]

pred_caso = modelo_nb.predict(caso)[0]
prob_caso = modelo_nb.predict_proba(caso)[0]
real_caso = y_test.iloc[i]

print("Clase real:", data.target_names[real_caso])
print("Clase predicha:", data.target_names[pred_caso])
print("Probabilidades:")

for clase, prob in zip(data.target_names, prob_caso):
    print(clase, ":", prob)

# 14. ¿Naive Bayes necesita escalado?

GaussianNB no depende de distancias como KNN o SVM.

Por eso el escalado no es tan crítico.

Aun así, en algunos flujos se puede escalar, pero no es obligatorio como en KNN/SVM.

In [ ]:
modelo_nb_escalado = Pipeline([
    ("scaler", StandardScaler()),
    ("model", GaussianNB())
])

modelo_nb_escalado.fit(X_train, y_train)
pred_nb_escalado = modelo_nb_escalado.predict(X_test)

pd.DataFrame([
    evaluar_clasificacion("Naive Bayes sin escalar", y_test, pred_nb),
    evaluar_clasificacion("Naive Bayes escalado", y_test, pred_nb_escalado)
])

# 15. Comparación contra otros modelos

Ahora comparamos Naive Bayes con modelos trabajados en el curso:

1. Regresión logística.
2. KNN.
3. Árbol.
4. Random Forest.
5. SVM.
6. Naive Bayes.

In [ ]:
modelo_logistica = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(max_iter=1000, random_state=42))
])

modelo_knn = Pipeline([
    ("scaler", StandardScaler()),
    ("model", KNeighborsClassifier(n_neighbors=5))
])

modelo_arbol = DecisionTreeClassifier(
    max_depth=4,
    random_state=42
)

modelo_rf = RandomForestClassifier(
    n_estimators=200,
    random_state=42
)

modelo_svm = Pipeline([
    ("scaler", StandardScaler()),
    ("model", SVC(kernel="rbf", C=1.0, gamma="scale", random_state=42))
])

modelos = {
    "Regresión logística": modelo_logistica,
    "KNN": modelo_knn,
    "Árbol": modelo_arbol,
    "Random Forest": modelo_rf,
    "SVM RBF": modelo_svm,
    "Naive Bayes": modelo_nb
}

resultados = []

for nombre, modelo in modelos.items():
    modelo.fit(X_train, y_train)
    pred = modelo.predict(X_test)
    resultados.append(evaluar_clasificacion(nombre, y_test, pred))

df_resultados = pd.DataFrame(resultados).sort_values("f1", ascending=False)

df_resultados

In [ ]:
metricas = ["accuracy", "precision", "recall", "f1"]

df_resultados.set_index("modelo")[metricas].plot(kind="bar", figsize=(12,5))
plt.title("Comparación de modelos supervisados")
plt.ylabel("Valor")
plt.ylim(0,1)
plt.xticks(rotation=25)
plt.grid(axis="y")
plt.show()

# 16. Interpretación de la comparación

Preguntas:

1. ¿Naive Bayes fue competitivo?
2. ¿Qué modelo obtuvo mejor F1?
3. ¿Qué modelo fue más interpretable?
4. ¿Qué modelo necesita escalado?
5. ¿Qué modelo usaría si necesita probabilidades rápidas?

Naive Bayes suele ser rápido, simple y útil como modelo base.

# 17. Ventajas y desventajas de Naive Bayes

## Ventajas

- Simple.
- Rápido.
- Funciona bien con pocos datos.
- Entrega probabilidades.
- Muy usado en clasificación de texto.
- Buen modelo base.

## Desventajas

- Supone independencia entre variables.
- Puede ser demasiado simple.
- La suposición de normalidad puede no cumplirse.
- Puede ser superado por modelos más flexibles.

# 18. ¿Cuándo usar Naive Bayes?

## Usar cuando:

- se necesita un modelo rápido,
- se requiere una línea base,
- hay clasificación de texto,
- se quieren probabilidades,
- el dataset no es enorme o se necesita eficiencia.

## Usar con cuidado cuando:

- las variables están muy correlacionadas,
- se requiere máxima precisión,
- las distribuciones no se parecen a las supuestas,
- se necesita capturar relaciones complejas.

# 19. Comparación conceptual

| Modelo | Enfoque |
|---|---|
| Regresión logística | probabilidad lineal |
| KNN | distancia |
| Árbol | reglas |
| Random Forest | ensamble |
| SVM | margen |
| Naive Bayes | probabilidad condicional |

Naive Bayes completa el panorama de clasificación porque introduce el enfoque probabilístico.

# 20. Taller autónomo

## Parte A — Conceptos

Responda:

1. ¿Qué es Naive Bayes?
2. ¿En qué teorema se basa?
3. ¿Qué significa probabilidad condicional?
4. ¿Por qué se llama naive?
5. ¿Qué suposición fuerte hace el modelo?
6. ¿Qué diferencia hay entre GaussianNB, MultinomialNB y BernoulliNB?

---

## Parte B — Implementación

1. Cargue el dataset.
2. Separe X e y.
3. Realice train/test.
4. Entrene GaussianNB.
5. Reporte accuracy, precision, recall y F1.
6. Genere matriz de confusión.

---

## Parte C — Probabilidades

1. Use `predict_proba`.
2. Seleccione 5 casos.
3. Muestre probabilidades por clase.
4. Explique cuándo el modelo está más seguro.

---

## Parte D — Comparación

Compare Naive Bayes contra:

1. Regresión logística.
2. KNN.
3. Árbol.
4. Random Forest.
5. SVM.

Responda:

1. ¿Cuál tuvo mejor F1?
2. ¿Naive Bayes fue competitivo?
3. ¿Qué modelo fue más simple?
4. ¿Cuál recomendaría y por qué?

---

## Parte E — Conclusión

Redacte mínimo 10 líneas indicando:

1. Qué aprendió de Naive Bayes.
2. Qué tan bien funcionó.
3. Qué ventajas tiene.
4. Qué limitaciones tiene.
5. En qué casos lo usaría.

# 21. Rúbrica sugerida

| Criterio | Puntaje |
|---|---:|
| Comprensión conceptual de Naive Bayes | 0.8 |
| Implementación correcta de GaussianNB | 0.8 |
| Evaluación con métricas y matriz de confusión | 0.8 |
| Interpretación de probabilidades | 0.8 |
| Comparación contra otros modelos | 1.0 |
| Conclusión técnica | 0.8 |
| **Total** | **5.0** |

---

# Cierre

Naive Bayes completa el bloque de clasificación probabilística.

Su idea central es:

> calcular la clase más probable dados los datos observados.

Aunque usa una suposición fuerte de independencia, suele ser un modelo rápido, simple y útil como línea base.